# Step 2: Feature Engineering & Aggregation
This notebook focuses on aggregating individual-level data into household-level features for both training and test sets.

In [5]:
import pandas as pd
import numpy as np
import plotly.express as px
import os

## 1. Defining Aggregation Logic

In [6]:
def aggregate_household_features(df):
    """Aggregate individual-level features to the household level."""
    ind_features = ['escolari', 'age', 'rez_esc', 'dis', 'male', 'female']
    
    agg_funcs = {
        'escolari': ['mean', 'max', 'min', 'std', 'sum'],
        'age': ['mean', 'max', 'min', 'std', 'sum'],
        'rez_esc': ['mean', 'max', 'sum'],
        'dis': ['sum', 'mean'], 
        'male': ['sum', 'mean'],
        'female': ['sum', 'mean']
    }
    
    house_agg = df.groupby('idhogar')[ind_features].agg(agg_funcs)
    house_agg.columns = ['_'.join(col).strip() for col in house_agg.columns.values]
    house_agg.reset_index(inplace=True)
    
    heads = df[df['parentesco1'] == 1].copy()
    final_df = heads.merge(house_agg, on='idhogar', how='left')
    
    return final_df

def create_derived_features(df):
    """Create new socio-economic indicators."""
    df['phones_per_person'] = df['qmobilephone'] / df['tamhog']
    df['tablets_per_person'] = df['v18q1'] / df['tamhog']
    df['rooms_per_person'] = df['rooms'] / df['tamhog']
    df['rent_per_person'] = df['v2a1'] / df['tamhog']
    df['children_ratio'] = df['hogar_nin'] / df['hogar_total']
    df['elderly_ratio'] = df['hogar_mayor'] / df['hogar_total']
    df['adult_ratio'] = df['hogar_adul'] / df['hogar_total']
    
    return df

## 2. Running the Aggregation
We process both the cleaned train and test files.

In [7]:
PROCESSED_PATH = '../data/processed/'

for split in ['train', 'test']:
    print(f"Processing {split} split...")
    df = pd.read_csv(os.path.join(PROCESSED_PATH, f'{split}_cleaned.csv'))
    df_eng = aggregate_household_features(df)
    df_eng = create_derived_features(df_eng)
    
    output_file = os.path.join(PROCESSED_PATH, f'{split}_engineered.csv')
    df_eng.to_csv(output_file, index=False)
    print(f"Saved to {output_file} (Shape: {df_eng.shape})")

print("Feature engineering complete!")

Processing train split...
Saved to ../data/processed/train_engineered.csv (Shape: (2973, 169))
Processing test split...
Saved to ../data/processed/test_engineered.csv (Shape: (7334, 168))
Feature engineering complete!
